In [48]:
# --- Nutrition vs Cost: Local USDA JSON + Open Prices API (single notebook cell) ---

import json
import statistics
import requests
from typing import Any, Dict, List, Optional

# =========================
# USER CONFIG (edit these)
# =========================
SEARCH_TERM = "pizza"          # <-- try: "beef hot dog", "whole milk", "apples"
USDA_JSON_PATH = "FoodData_Central_foundation_food_json_2025-12-18.json"  # <-- local USDA dump (your file)
OPEN_PRICES_BASE = "https://prices.openfoodfacts.org/api/v1"
TIMEOUT = 30

# =========================
# HELPERS
# =========================
def safe_float(x: Any) -> Optional[float]:
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def http_get_json(url: str, params: Optional[Dict[str, Any]] = None) -> Any:
    r = requests.get(url, params=params, timeout=TIMEOUT, headers={
        "User-Agent": "NutritionVsCostStudentProject/1.0 (educational use)"
    })
    r.raise_for_status()
    return r.json()

def load_usda_json(path: str) -> List[Dict[str, Any]]:
    """Load USDA JSON dump (either a single object or a list of objects)."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    if isinstance(data, dict):
        return [data]
    if isinstance(data, list):
        return data
    raise ValueError("USDA JSON must be a dict or a list of dicts")

# =========================
# 1) USDA LOCAL: MATCH + EXTRACT NUTRITION
# =========================
from typing import Any, Dict, List, Optional

def find_best_usda_food(
    foods: List[Dict[str, Any]],
    term: str
) -> Dict[str, Any]:
    """
    Simple match: find the first USDA item whose description contains the term
    (case-insensitive).
    Fallback: first record.
    """
    if not foods:
        raise RuntimeError("No USDA foods available in the dump")

    term_l = term.lower().strip()

    if term_l:
        for food in foods:
            desc = (food.get("description") or "").lower()

            # ✅ Only take the first description part before the comma
            desc_head = desc.split(",", 1)[0].strip()

            if term_l and term_l in desc_head:
                return food

    return foods[0]


def extract_usda_nutrition(
    food: Dict[str, Any]
) -> Dict[str, Any]:
    """
    Extract calories and protein per 100 g, plus serving grams (piece) if available.

    If no serving is available, defaults serving_g = 100
    (so metrics are per-100g).
    """
    calories_100g: Optional[float] = None
    protein_100g: Optional[float] = None

    for fn in food.get("foodNutrients") or []:
        nutrient = fn.get("nutrient") or {}
        name = (nutrient.get("name") or "").lower()
        unit = (nutrient.get("unitName") or "").lower()
        amount = safe_float(fn.get("amount"))

        if amount is None:
            continue

        if name == "energy" and unit == "kcal":
            calories_100g = amount
        elif name == "protein" and unit == "g":
            protein_100g = amount

    # Find serving grams for a "piece" if present
    serving_g: Optional[float] = None

    for portion in food.get("foodPortions") or []:
        mu = (portion.get("measureUnit") or {}).get("name")
        if mu and mu.lower() == "piece":
            serving_g = safe_float(portion.get("gramWeight"))
            if serving_g is not None:
                break

    if serving_g is None:
        serving_g = 100.0

    if calories_100g is None or protein_100g is None:
        raise RuntimeError(
            "USDA nutrition missing Energy (kcal) or Protein (g)"
        )

    return {
        "food_name": food.get("description"),
        "fdc_id": food.get("fdcId"),
        "calories_100g": calories_100g,
        "protein_100g": protein_100g,
        "serving_g": serving_g,
    }


def get_usda_nutrition_local(
    term: str,
    usda_json_path: str
) -> Dict[str, Any]:
    foods = load_usda_json(usda_json_path)[0]["FoundationFoods"]
    print(foods[0])
    print(type(foods))        # list
    print(len(foods))         # number of food entries

    best_food = find_best_usda_food(foods, term)
    return extract_usda_nutrition(best_food)
# =========================
# 2) OPEN PRICES API: SEARCH PRODUCTS + FETCH PRICES
# =========================
def openprices_search_products(term: str, size: int = 50, page: int = 1) -> List[Dict[str, Any]]:
    """
    Search Open Prices products by product_name__like (returns barcodes in 'code').
    """
    url = f"{OPEN_PRICES_BASE}/products"
    
    search_term = term.split(",", 1)[0].strip()

    params = {"product_name__like": search_term, "size": size, "page": page}
    data = http_get_json(url, params=params)

    items = data.get("items", [])
    # Sometimes APIs return nested items; flatten one level if needed
    if items and isinstance(items[0], list):
        items = items[0]
    return [x for x in items if isinstance(x, dict)]

def openprices_get_prices(product_code: str) -> List[float]:
    """
    Get USD price-per-item for a given barcode using Open Prices /prices endpoint.
    price_per_item = price / receipt_quantity
    """
    url = f"{OPEN_PRICES_BASE}/prices"
    print(url)
    params = {"product_code": product_code}
    print(params)
    data = http_get_json(url, params=params)

    records = []
    if isinstance(data, dict):
        records = data.get("results", data.get("items", []))
    elif isinstance(data, list):
        records = data

    prices_usd_per_item = []
    for rec in records:
        if not isinstance(rec, dict):
            continue
        if rec.get("currency") != "USD":
            continue
        price = safe_float(rec.get("price"))
        qty = safe_float(rec.get("receipt_quantity")) or 1.0
        if price is not None and qty > 0:
            prices_usd_per_item.append(price / qty)

    return prices_usd_per_item

def average_price_usd(term: str, max_products: int = 100, product_page_size: int = 100) -> Dict[str, Any]:
    """
    1) Search products by term
    2) Take up to max_products barcodes
    3) Pull all USD prices for each barcode
    4) Return mean + median
    """
    products = openprices_search_products(term, size=product_page_size, page=1)
    codes = [p.get("code") for p in products if (p.get("code") is not None) & (p.get('price_count') > 0)]
    codes = [str(c) for c in codes][:max_products]

    all_prices = []
    per_code_counts = {}
    for code in codes:
        vals = openprices_get_prices(code)
        if vals:
            per_code_counts[code] = len(vals)
            all_prices.extend(vals)

    if not all_prices:
        return {
            "avg_price_usd": None,
            "median_price_usd": None,
            "n_prices": 0,
            "n_products_considered": len(codes),
            "per_code_counts": per_code_counts
        }

    return {
        "avg_price_usd": statistics.mean(all_prices),
        "median_price_usd": statistics.median(all_prices),
        "n_prices": len(all_prices),
        "n_products_considered": len(codes),
        "per_code_counts": per_code_counts
    }

# =========================
# 3) COMBINE: NUTRITION vs COST
# =========================
def nutrition_vs_cost(term: str, usda_json_path: str) -> Dict[str, Any]:
    nutrition = get_usda_nutrition_local(term, usda_json_path)
    pricing = average_price_usd(term)

    if pricing["avg_price_usd"] is None:
        return {
            "search_term": term,
            "error": "No USD prices found for matching products in Open Prices.",
            "usda_match": nutrition,
            "price_summary": pricing
        }

    avg_price = pricing["avg_price_usd"]

    calories_per_item = nutrition["calories_100g"] * (nutrition["serving_g"] / 100.0)
    protein_per_item = nutrition["protein_100g"] * (nutrition["serving_g"] / 100.0)

    return {
        "search_term": term,
        "sources": {
            "nutrition": f"Local USDA JSON dump: {usda_json_path}",
            "prices": "Open Prices API (prices.openfoodfacts.org)"
        },
        "usda_match": nutrition,
        "price_summary_usd": {
            "avg_price_per_item_usd": round(avg_price, 4),
            "median_price_per_item_usd": round(pricing["median_price_usd"], 4),
            "n_price_points_usd": pricing["n_prices"],
            "n_products_considered": pricing["n_products_considered"]
        },
        "computed_metrics": {
            "calories_per_item": round(calories_per_item, 2),
            "protein_g_per_item": round(protein_per_item, 2),
            "calories_per_usd": round(calories_per_item / avg_price, 2),
            "protein_g_per_usd": round(protein_per_item / avg_price, 2),
        }
    }

# =========================
# RUN
# =========================
result = nutrition_vs_cost(SEARCH_TERM, USDA_JSON_PATH)
print(json.dumps(result, indent=2))

{'foodClass': 'FinalFood', 'description': 'Hummus, commercial', 'foodNutrients': [{'type': 'FoodNutrient', 'id': 2219707, 'nutrient': {'id': 1120, 'number': '334', 'name': 'Cryptoxanthin, beta', 'rank': 7460, 'unitName': 'µg'}, 'dataPoints': 1, 'foodNutrientDerivation': {'code': 'A', 'description': 'Analytical', 'foodNutrientSource': {'id': 1, 'code': '1', 'description': 'Analytical or derived from analytical'}}, 'median': 3.0, 'amount': 3.0}, {'type': 'FoodNutrient', 'id': 2219708, 'nutrient': {'id': 1122, 'number': '337', 'name': 'Lycopene', 'rank': 7530, 'unitName': 'µg'}, 'dataPoints': 1, 'foodNutrientDerivation': {'code': 'A', 'description': 'Analytical', 'foodNutrientSource': {'id': 1, 'code': '1', 'description': 'Analytical or derived from analytical'}}, 'median': 0.0, 'amount': 0.0}, {'type': 'FoodNutrient', 'id': 2219709, 'nutrient': {'id': 1127, 'number': '343', 'name': 'Tocopherol, delta', 'rank': 8200, 'unitName': 'mg'}, 'dataPoints': 6, 'foodNutrientDerivation': {'code': '